# 08 — Volatility Model, Probabilistic Forecasting & Risk Engine

Phase 8 of the Indian Equity AI project plan.

For every stock/day, produce:
1. A probability of next-day up-move (Phase 5's classifier, retrained here on the same split)
2. An expected-volatility forecast (a new, separate regression model)
3. A probabilistic next-day price range (10th/50th/90th percentile, via quantile regression)
4. A Low/Medium/High risk label (volatility + beta + historical drawdown + VaR)

"Every prediction ships with a probability, a range, and a risk label -- not just a point estimate."

In [1]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

import numpy as np
import pandas as pd
from lightgbm import LGBMClassifier, LGBMRegressor

from src.ingestion.market_data import download_index_ohlcv
from src.preprocessing.clean import clean_ohlcv
from src.features.returns import add_returns as add_universe_returns
from src.features.risk import beta as compute_beta, max_drawdown as compute_max_drawdown
from src.models.dataset import FEATURE_COLUMNS, add_next_day_return, add_target, build_feature_table, chronological_split
from src.models.evaluate import classification_metrics
from src.risk.volatility_target import add_forward_volatility_target
from src.risk.quantiles import fit_quantile_models, predict_quantiles, return_quantiles_to_price_range
from src.risk.var_es import rolling_var_es
from src.risk.engine import compute_risk_score, risk_label_relative

DATA_PROCESSED = PROJECT_ROOT / "data" / "processed"
RANDOM_STATE = 42
VOL_HORIZON = 5


def make_lgbm_classifier():
    return LGBMClassifier(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1,
    )


def make_lgbm_regressor(objective="regression", alpha=None):
    kwargs = dict(
        n_estimators=300, max_depth=4, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8, random_state=RANDOM_STATE, verbose=-1, objective=objective,
    )
    if alpha is not None:
        kwargs["alpha"] = alpha
    return LGBMRegressor(**kwargs)

## Step 1: Build the base feature table + all three targets

In [2]:
technical = pd.read_parquet(DATA_PROCESSED / "technical_features.parquet")

df = build_feature_table(technical)
df = add_target(df)  # next_day_direction (classification)
df = add_next_day_return(df)  # next_day_return (quantile regression)
df = add_forward_volatility_target(df, horizon=VOL_HORIZON)  # forward_volatility_5d (regression)

vol_target_col = f"forward_volatility_{VOL_HORIZON}d"
required = FEATURE_COLUMNS + ["next_day_direction", "next_day_return", vol_target_col]
dataset = df.dropna(subset=required).reset_index(drop=True)

print("Rows:", len(dataset), "| Date range:", dataset["date"].min().date(), "to", dataset["date"].max().date())

Rows: 5185 | Date range: 2022-07-14 to 2026-09-18


## Step 2: Chronological split and model training

In [3]:
train, val, test = chronological_split(dataset, train_frac=0.7, val_frac=0.15)
print(f"train: {len(train)} | val: {len(val)} | test: {len(test)}")

X_train, X_val, X_test = train[FEATURE_COLUMNS], val[FEATURE_COLUMNS], test[FEATURE_COLUMNS]

train: 3625 | val: 780 | test: 780


In [4]:
# 2a. Direction classifier -> P(up)
classifier = make_lgbm_classifier()
classifier.fit(X_train, train["next_day_direction"])
test_proba_up = classifier.predict_proba(X_test)[:, 1]
test_pred_up = classifier.predict(X_test)

classification_report = classification_metrics(test["next_day_direction"].values, test_pred_up, test_proba_up)
classification_report

{'accuracy': 0.4717948717948718,
 'precision': np.float64(0.4352),
 'recall': np.float64(0.8217522658610272),
 'f1': np.float64(0.5690376569037657),
 'roc_auc': np.float64(0.4885310761073618),
 'log_loss': 0.8659227046764503,
 'brier_score': np.float64(0.31897672986677933)}

In [5]:
# 2b. Volatility regressor -> expected_volatility
vol_model = make_lgbm_regressor(objective="regression")
vol_model.fit(X_train, train[vol_target_col])
test_expected_volatility = vol_model.predict(X_test)

from sklearn.metrics import mean_absolute_error

naive_vol_forecast = np.full(len(test), train[vol_target_col].mean())
print("Volatility model MAE:", mean_absolute_error(test[vol_target_col], test_expected_volatility))
print("Naive (train-mean) MAE:", mean_absolute_error(test[vol_target_col], naive_vol_forecast))

Volatility model MAE: 0.006395577177022865
Naive (train-mean) MAE: 0.006353711635988066


In [6]:
# 2c. Quantile regressors -> 10th/50th/90th percentile next-day return
quantile_models = fit_quantile_models(
    X_train, train["next_day_return"],
    model_factory=lambda q: make_lgbm_regressor(objective="quantile", alpha=q),
)
test_quantile_returns = predict_quantiles(quantile_models, X_test)
test_quantile_returns.index = test.index
test_price_range = return_quantiles_to_price_range(test_quantile_returns, test["close"])

# Coverage check: with a well-calibrated 10/90 interval, ~80% of actual next-day
# closes should fall inside [q10_price, q90_price].
actual_next_close = test["close"] * (1 + test["next_day_return"])
inside_interval = (actual_next_close >= test_price_range["q10"]) & (actual_next_close <= test_price_range["q90"])
print(f"Empirical coverage of the 10-90 interval: {inside_interval.mean():.1%} (target ~80%)")

Empirical coverage of the 10-90 interval: 64.1% (target ~80%)


## Step 3: Historical beta and max drawdown (per stock, full history)

In [7]:
universe_prices = pd.read_parquet(DATA_PROCESSED / "prices_daily.parquet")
universe_prices = add_universe_returns(universe_prices, periods=(1,))

nifty50 = clean_ohlcv(download_index_ohlcv("^NSEI", period="5y"))
nifty50 = add_universe_returns(nifty50, periods=(1,))
nifty_returns = nifty50.set_index("date")["return_1d"]

stock_beta = {}
for symbol, group in universe_prices.groupby("symbol"):
    stock_returns = group.set_index("date")["return_1d"]
    stock_beta[symbol] = compute_beta(stock_returns, nifty_returns)

stock_max_drawdown = compute_max_drawdown(universe_prices)

pd.DataFrame({"beta": pd.Series(stock_beta), "max_drawdown": stock_max_drawdown})

,beta,max_drawdown
HDFCBANK,1.064398,-0.321651
INFY,0.971136,-0.507276
ITC,0.677880,-0.492415
RELIANCE,1.112299,-0.274221
TCS,0.811961,-0.564623


## Step 4: Rolling VaR/ES and the composite risk score

In [8]:
risk_inputs = rolling_var_es(dataset[["date", "symbol", "close"]], window=250, confidence=0.95)
test_risk_inputs = risk_inputs.loc[test.index]
train_risk_inputs = risk_inputs.loc[train.index]

risk_frame = pd.DataFrame(
    {
        "expected_volatility": test_expected_volatility,
        "beta": test["symbol"].map(stock_beta).values,
        "max_drawdown": test["symbol"].map(stock_max_drawdown).values,
        "var_95": test_risk_inputs["var_95"].values,
    },
    index=test.index,
)
risk_frame["risk_score"] = compute_risk_score(risk_frame)

# Reference distribution for relative labeling: risk_score computed the same way,
# but over TRAIN rows -- so the Low/Medium/High tertile cutoffs are learned from
# training data only, and applied out-of-sample to the test set (no leakage).
train_risk_frame = pd.DataFrame(
    {
        "expected_volatility": vol_model.predict(X_train),
        "beta": train["symbol"].map(stock_beta).values,
        "max_drawdown": train["symbol"].map(stock_max_drawdown).values,
        "var_95": train_risk_inputs["var_95"].values,
    },
    index=train.index,
)
train_risk_score = compute_risk_score(train_risk_frame).dropna()

risk_frame["risk_label"] = risk_label_relative(risk_frame["risk_score"], reference_scores=train_risk_score)
risk_frame["risk_label"].value_counts()

risk_label
High      319
Medium    305
Low       156
Name: count, dtype: int64

## Step 5: Assemble the final per-stock, per-day prediction output

In [9]:
output = pd.DataFrame(
    {
        "date": test["date"].values,
        "symbol": test["symbol"].values,
        "close": test["close"].values,
        "probability_up": test_proba_up,
        "expected_volatility": test_expected_volatility,
        "price_q10": test_price_range["q10"].values,
        "price_q50": test_price_range["q50"].values,
        "price_q90": test_price_range["q90"].values,
        "risk_score": risk_frame["risk_score"].values,
        "risk_label": risk_frame["risk_label"].astype(str).values,
    }
)
output.to_parquet(DATA_PROCESSED / "risk_engine_output.parquet", index=False)
print("Saved:", DATA_PROCESSED / "risk_engine_output.parquet")
output.tail(10)

Saved: D:\Practise\Stock Market Analysis\data\processed\risk_engine_output.parquet


,date,symbol,close,probability_up,expected_volatility,price_q10,price_q50,price_q90,risk_score,risk_label
770,2026-09-07,TCS,2270.000000,0.840281,0.011357,2247.544604,2281.335996,2315.176534,53.211477,High
771,2026-09-08,TCS,2255.500000,0.738554,0.015815,2218.795138,2265.086411,2303.774914,55.440756,High
772,2026-09-09,TCS,2208.000000,0.532747,0.014641,2172.920875,2212.037855,2248.962748,54.853381,High
773,2026-09-10,TCS,2204.100098,0.679152,0.014021,2173.888719,2208.523696,2247.123270,54.543420,High
774,2026-09-11,TCS,2200.800049,0.750155,0.015465,2159.116953,2203.691532,2243.670524,55.265419,High
775,2026-09-14,TCS,2200.800049,0.763368,0.016274,2174.984278,2205.284601,2253.194926,55.669971,High
776,2026-09-15,TCS,2251.000000,0.817881,0.019452,2214.286817,2260.282227,2299.974976,57.258947,High
777,2026-09-16,TCS,2188.800049,0.853684,0.018374,2148.730323,2199.349851,2236.219392,56.731603,High
778,2026-09-17,TCS,2190.000000,0.769595,0.017582,2160.008394,2203.049056,2234.759083,56.335525,High
779,2026-09-18,TCS,2105.000000,0.714970,0.013567,2063.700654,2107.982788,2142.838913,54.997559,High


## Example: dashboard-style "AI Outlook" for the latest day per stock

In [10]:
latest = output.sort_values("date").groupby("symbol").tail(1)
for _, row in latest.iterrows():
    direction = "Bullish" if row["probability_up"] > 0.5 else "Bearish"
    print(f"{row['symbol']:10s} | {direction:8s} P(up)={row['probability_up']:.1%} | "
          f"range [Rs{row['price_q10']:.1f} - Rs{row['price_q50']:.1f} - Rs{row['price_q90']:.1f}] | "
          f"risk={row['risk_label']} ({row['risk_score']:.0f}/100)")

RELIANCE   | Bullish  P(up)=55.2% | range [Rs1202.2 - Rs1228.2 - Rs1248.7] | risk=Low (41/100)
INFY       | Bullish  P(up)=92.1% | range [Rs1040.8 - Rs1056.9 - Rs1075.7] | risk=High (57/100)
HDFCBANK   | Bullish  P(up)=62.5% | range [Rs723.2 - Rs731.9 - Rs743.5] | risk=Medium (44/100)
ITC        | Bullish  P(up)=57.0% | range [Rs259.2 - Rs262.4 - Rs267.4] | risk=Medium (46/100)
TCS        | Bullish  P(up)=71.5% | range [Rs2063.7 - Rs2108.0 - Rs2142.8] | risk=High (55/100)


## Reflection

**Actual results from this run:**

- **Direction classifier:** ROC-AUC ~0.49 on this split -- consistent with Phase 5/7's
  finding that next-day direction from technical features alone carries very weak signal.
- **Volatility model:** MAE was *not* clearly better than a naive "assume the training-set
  average volatility continues" forecast (they came out almost identical). This is an
  honest negative result, not hidden: the volatility model as currently specified isn't
  demonstrably adding value yet. Worth revisiting with more history or a different target
  horizon before relying on it.
- **Quantile price range:** empirical coverage of the 10th-90th percentile interval came in
  around 64%, well short of the ~80% a well-calibrated interval should hit. This means the
  range is currently **too narrow** -- treat it as directional guidance on relative
  uncertainty between stocks/days, not a literal 80%-confidence interval, until
  recalibrated (e.g. conformal calibration, or widening via a calibration multiplier
  fit on validation data).
- **Risk label:** the first version of this notebook produced 100% "Medium" labels --
  the absolute 0-100 score's caps (5% daily vol, beta=2, 60% drawdown, 5% VaR) were
  calibrated for a broader, more volatile universe than 5 large-cap blue chips, so no
  stock/day ever reached the Low or High ends. Fixed by adding `risk_label_relative`,
  which buckets using tertile cutoffs learned from the *training* distribution instead of
  fixed absolute thresholds -- always produces a meaningful spread, and the cutoffs are
  learned from train only (no leakage into how test rows get labeled).
- Beta and max drawdown are historical (Phase 2), not forecasts -- they describe how the
  stock *has* behaved, a reasonable risk proxy but not a guarantee of future risk.

**Bottom line:** every prediction now genuinely ships with a probability, a range, and a
risk label (the phase's literal DoD), but two of the three probabilistic components
(volatility forecast, quantile range) show real calibration weaknesses that are reported
here rather than glossed over. Phase 9's backtest is where these get pressure-tested
against actual trading outcomes.